# Income vs. Transit Stop Density — Statistical Test

Is a census tract's median household income related to how many transit stops it has per km²?

We join the per-tract **income distribution** and **transit stop density** tables on `GEOID` and run formal statistical tests to decide whether the relationship is real or could plausibly be noise.

**Inputs**
- `output_csvs/san_diego_income_distribution.csv` &mdash; one row per tract, with `median_hh_income`.
- `output_csvs/san_diego_transit_stop_density.csv` &mdash; one row per tract, with `stops_per_km2`.

**Tests we run**
1. **Pearson** correlation — strength of a *linear* association (with p-value).
2. **Spearman** rank correlation — strength of a *monotonic* association, robust to skew and outliers (with p-value).
3. **OLS regression** — slope estimate, R², and the slope's p-value.
4. **Group comparison** — split tracts at the median income and test whether transit density differs between the lower- and higher-income halves (Welch t-test + Mann–Whitney U).

Throughout we use a significance threshold of **α = 0.05**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import warnings
warnings.simplefilter(action='ignore', category=Warning)

pd.set_option('display.max_columns', None)
ALPHA = 0.05

## 1. Load and join the two tables

Both tables are keyed by `GEOID` (the census-tract id). We do an inner join so every row has both an income and a density value, then drop any rows where either value is missing.

In [ ]:
income = pd.read_csv('output_csvs/san_diego_income_distribution.csv')
density = pd.read_csv('output_csvs/san_diego_transit_stop_density.csv')

df = income.merge(density[['GEOID', 'num_stops', 'area_km2', 'stops_per_km2']], on='GEOID', how='inner')
df = df.dropna(subset=['median_hh_income', 'stops_per_km2'])

print(f'{len(income):,} income tracts × {len(density):,} density tracts → {len(df):,} matched tracts')
df[['GEOID', 'area', 'median_hh_income', 'num_stops', 'stops_per_km2']].head()

## 2. Look at the variables before testing

Both `median_hh_income` and `stops_per_km2` are right-skewed (a few very high-income tracts; a few very dense tracts). Pearson's test assumes roughly bivariate-normal data, so skew is a reason to also trust the rank-based Spearman test and to consider a log transform.

In [ ]:
df[['median_hh_income', 'stops_per_km2']].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df['median_hh_income'].dropna(), bins=40, color='#4c78a8')
axes[0].set_title('Median household income')
axes[0].set_xlabel('USD')
axes[1].hist(df['stops_per_km2'].dropna(), bins=40, color='#4c78a8')
axes[1].set_title('Transit stops per km²')
axes[1].set_xlabel('stops / km²')
for ax in axes:
    ax.set_ylabel('# tracts')
plt.tight_layout()
plt.show()

## 3. Correlation tests (the main p-values)

- **Pearson r** measures linear association.
- **Spearman ρ** measures monotonic association on ranks — it doesn't care about the skew, so it's the more defensible test here.

For each, the **p-value** is the probability of seeing a correlation at least this strong if income and transit density were truly unrelated (the null hypothesis).

In [ ]:
x = df['median_hh_income'].to_numpy()
y = df['stops_per_km2'].to_numpy()

pearson = stats.pearsonr(x, y)
spearman = stats.spearmanr(x, y)

results = pd.DataFrame({
    'test': ['Pearson (linear)', 'Spearman (rank/monotonic)'],
    'statistic': [pearson.statistic, spearman.statistic],
    'p_value': [pearson.pvalue, spearman.pvalue],
})
results['significant_at_0.05'] = results['p_value'] < ALPHA
results

In [ ]:
for _, row in results.iterrows():
    direction = 'positive' if row['statistic'] > 0 else 'negative'
    verdict = 'REJECT the null — the association is statistically significant' if row['p_value'] < ALPHA \
        else 'fail to reject the null — no significant association'
    print(f"{row['test']:<28} stat={row['statistic']:+.4f} ({direction}), p={row['p_value']:.3e}")
    print(f"    → {verdict}\n")

## 4. Scatter plot with regression line

A visual check of what the correlation coefficient is summarizing.

In [ ]:
slope, intercept, r, p, se = stats.linregress(x, y)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x, y, s=14, alpha=0.4, color='#4c78a8', edgecolor='none')
xs = np.linspace(x.min(), x.max(), 100)
ax.plot(xs, intercept + slope * xs, color='#e4572e', lw=2,
        label=f'OLS fit (R²={r**2:.3f}, p={p:.2e})')
ax.set_xlabel('Median household income (USD)')
ax.set_ylabel('Transit stops per km²')
ax.set_title('Income vs. transit stop density by census tract')
ax.legend()
plt.tight_layout()
plt.show()

print(f'slope        = {slope:.6e} stops/km² per $1 of income')
print(f'per $10k     = {slope*10_000:+.3f} stops/km²')
print(f'R²           = {r**2:.4f}')
print(f'slope p-value= {p:.3e}')

## 5. Group comparison: low- vs. high-income tracts

Split tracts into two halves at the **median** income and ask: do the halves have different transit density?

- **Welch's t-test** compares the means (does not assume equal variances).
- **Mann–Whitney U** compares distributions on ranks — the non-parametric backup, appropriate given the skew.

In [ ]:
median_income = df['median_hh_income'].median()
low = df[df['median_hh_income'] <= median_income]['stops_per_km2']
high = df[df['median_hh_income'] > median_income]['stops_per_km2']

ttest = stats.ttest_ind(low, high, equal_var=False)
mwu = stats.mannwhitneyu(low, high, alternative='two-sided')

print(f'Median income split point: ${median_income:,.0f}')
print(f'  Lower-income half : n={len(low):>3}, mean density = {low.mean():.3f} stops/km² (median {low.median():.3f})')
print(f'  Higher-income half: n={len(high):>3}, mean density = {high.mean():.3f} stops/km² (median {high.median():.3f})')
print()
print(f"Welch's t-test : t={ttest.statistic:+.3f}, p={ttest.pvalue:.3e}")
print(f'Mann–Whitney U : U={mwu.statistic:.0f}, p={mwu.pvalue:.3e}')
for name, pval in [('Welch t-test', ttest.pvalue), ('Mann–Whitney', mwu.pvalue)]:
    print(f'    {name}: {"significant" if pval < ALPHA else "not significant"} at α={ALPHA}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.boxplot([low.dropna(), high.dropna()], labels=['Lower income', 'Higher income'], showfliers=False)
ax.set_ylabel('Transit stops per km²')
ax.set_title('Transit density by income half')
plt.tight_layout()
plt.show()

## 6. Summary

Run the cell below for a one-line readout of every test. Interpretation guide:

- A **small p-value (< 0.05)** means the observed relationship is unlikely under the "no relationship" null — we call it statistically significant.
- The **sign** of the correlation / slope tells you the direction: positive = higher-income tracts tend to have *more* stops per km²; negative = *fewer*.
- Statistical significance is **not** the same as a large effect — check R² and the per-$10k slope to judge whether the relationship is practically meaningful.

In [ ]:
summary = pd.DataFrame([
    {'test': 'Pearson correlation',  'statistic': pearson.statistic,  'p_value': pearson.pvalue},
    {'test': 'Spearman correlation', 'statistic': spearman.statistic, 'p_value': spearman.pvalue},
    {'test': 'OLS slope',            'statistic': slope,              'p_value': p},
    {'test': 'Welch t-test',         'statistic': ttest.statistic,    'p_value': ttest.pvalue},
    {'test': 'Mann–Whitney U',       'statistic': mwu.statistic,      'p_value': mwu.pvalue},
])
summary['significant'] = summary['p_value'] < ALPHA
summary